# Adaptive KDE & WRx vs IRx Classification

Implements Sections 2.3–2.4 of the manuscript:

> We used Adaptive Kernel Density Estimation (KDE), which converts discrete rainfall observations into
> continuous density surfaces... We then applied Adaptive KDE to this fixed network from 1880 onwards... For the
> earlier period (pre-1880), we used the available station network... We assigned each HRD a spatial ratio value
> ... We classified a HRD as an IRx day if its spatial ratio fell below 16%, and as a WRx day if the ratio reached
> 16% or higher... We grouped consecutive WRx days and treated them as a single widespread heavy rainfall event
> ... We further classified WRx events into coastal, inland and mixed categories.

**Confirmed parameters:**
- Blob threshold = 75th percentile of the daily Adaptive KDE surface
- Stations included in the KDE if `Rainfall > 0`

**Pipeline:**
1. Adaptive KDE + spatial-ratio function (shared by both periods)
2. Apply to pre-1880 HRDs using the full available network
3. Apply to 1880–2024 HRDs using the constant network (`Constant_Station_Network_clean.ipynb` output)
4. Merge into one ratio series and derive the 16% WRx/IRx threshold
5. Classify HRDs as WRx / IRx
6. Group consecutive WRx (and IRx) days into events
7. Classify WRx events as Coastal / Inland / Mixed
8. Weekday (Monday/weekend) bias QC check

## Step 0 — Config

In [ ]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.ops import unary_union
from sklearn.neighbors import KernelDensity
from scipy.stats import chi2_contingency, binomtest
from tqdm import tqdm

# ---- Paths (edit these) ----------------------------------------------------
FULL_NETWORK_FILE      = "/path/to/combined_parquet.parquet"        # all 5,429 stations, pre-1880 period
CONSTANT_NETWORK_FILE  = "/path/to/Trend_study/combined_parquet.parquet"  # output of Constant_Station_Network_clean.ipynb
HRD_DATES_FILE         = "/path/to/HRD_dates.xlsx"                  # output of HRD_Definition_clean.ipynb
GRID_FILE              = "/path/to/grid_0p001.geojson"              # 0.1 deg grid clipped to NSW
NSW_BOUNDARY_FILE      = "/path/to/Interest_map_with_GDR.geojson"
COASTAL_POLYGON_FILE   = "/path/to/coastal_polygons.geojson"

OUTPUT_FILE = "/path/to/WRx_IRx_classification.xlsx"

# ---- Parameters (confirmed) -------------------------------------------------
FIXED_NETWORK_START_YEAR = 1880
BLOB_PERCENTILE          = 75      # blob = grid cells with KDE >= this percentile of the day's KDE surface
MIN_RAINFALL_MM          = 0.0     # station included in KDE if Rainfall > this
KDE_BASE_BANDWIDTH        = 0.5
EPS                       = 1e-6

## Step 1 — Adaptive KDE + spatial-ratio function

One function, reused for both the pre-1880 (full network) and 1880+ (constant network) periods — this was
duplicated with minor drift across your original notebook's cells 11–12, 17–18, and 28–29; consolidated here
into a single, parametrised version.

In [ ]:
def compute_daily_blob_ratio(daily_data, grid_gdf, grid_coords, total_area_km2):
    """Adaptive KDE for a single day's station rainfall -> (blob_area_km2, blob_ratio, centroid_lat, centroid_lon).
    daily_data must have columns Lat, Lon, Rainfall and already be filtered to Rainfall > MIN_RAINFALL_MM."""
    if daily_data.empty:
        return np.nan, np.nan, np.nan, np.nan

    coords = np.vstack([daily_data["Lat"].values, daily_data["Lon"].values]).T
    rainfall = daily_data["Rainfall"].values

    # --- Pilot KDE (sets adaptive bandwidths) ---
    pilot_kde = KernelDensity(bandwidth=KDE_BASE_BANDWIDTH, kernel="gaussian")
    pilot_kde.fit(coords, sample_weight=rainfall)
    pilot_density = np.maximum(np.exp(pilot_kde.score_samples(coords)), EPS)
    g = np.exp(np.mean(np.log(pilot_density)))
    scales = (pilot_density / g) ** -0.5

    # --- Adaptive KDE on the grid ---
    adaptive_values = np.zeros(len(grid_coords))
    for i in range(len(coords)):
        bw_i = KDE_BASE_BANDWIDTH * scales[i]
        kde_i = KernelDensity(bandwidth=bw_i, kernel="gaussian")
        kde_i.fit(coords[i:i + 1], sample_weight=[rainfall[i]])
        adaptive_values += np.exp(kde_i.score_samples(grid_coords))

    thr_value = np.nanpercentile(adaptive_values, BLOB_PERCENTILE)
    grid_gdf = grid_gdf.copy()
    grid_gdf["Blob"] = adaptive_values >= thr_value
    blob_cells = grid_gdf[grid_gdf["Blob"]]

    if blob_cells.empty:
        return np.nan, np.nan, np.nan, np.nan

    blob_area_km2 = blob_cells.to_crs(epsg=3577).geometry.area.sum() / 1e6
    blob_ratio = blob_area_km2 / total_area_km2

    unioned = unary_union(blob_cells.geometry)
    polygons = [unioned] if unioned.geom_type == "Polygon" else [p for p in unioned.geoms if p.area > 0]
    largest_blob = max(polygons, key=lambda p: p.area) if polygons else None
    if largest_blob is not None:
        centroid = largest_blob.centroid
        centroid_lat, centroid_lon = centroid.y, centroid.x
    else:
        centroid_lat = centroid_lon = np.nan

    del adaptive_values, pilot_density, scales
    gc.collect()
    return blob_area_km2, blob_ratio, centroid_lat, centroid_lon


def run_blob_ratio_for_period(hrd_df, station_data_file, nsw_boundary_file, grid_file):
    """Run compute_daily_blob_ratio for every HRD date in hrd_df, using the given station data file."""
    stations_df = pd.read_parquet(station_data_file)
    stations_df["Date"] = pd.to_datetime(stations_df["Date"], errors="coerce")

    gdf_map = gpd.read_file(nsw_boundary_file)
    grid_gdf = gpd.read_file(grid_file)
    total_area_km2 = gdf_map.to_crs(epsg=3577).geometry.area.sum() / 1e6

    grid_centroids = grid_gdf.copy()
    grid_centroids["geometry"] = grid_gdf.centroid
    grid_coords = np.array([[pt.y, pt.x] for pt in grid_centroids.geometry])

    results = []
    for date in tqdm(hrd_df["Date"], desc="Processing HRDs"):
        daily_data = stations_df[
            (stations_df["Date"] == pd.to_datetime(date)) & (stations_df["Rainfall"] > MIN_RAINFALL_MM)
        ]
        area, ratio, c_lat, c_lon = compute_daily_blob_ratio(daily_data, grid_gdf, grid_coords, total_area_km2)
        results.append({
            "Date": date, "Blob_Area_km2": area, "Blob_Ratio": ratio,
            "Blob_Centroid_Lat": c_lat, "Blob_Centroid_Lon": c_lon,
        })
    return pd.DataFrame(results)

## Step 2 — Apply to each period

Pre-1880 (1858–1879): full available network. 1880–2024: constant network (from
`Constant_Station_Network_clean.ipynb`).

In [ ]:
hrd_df = pd.read_excel(HRD_DATES_FILE)
hrd_df["Date"] = pd.to_datetime(hrd_df["Date"], errors="coerce")

pre_1880_hrds = hrd_df[hrd_df["Date"] < f"{FIXED_NETWORK_START_YEAR}-01-01"].reset_index(drop=True)
post_1880_hrds = hrd_df[hrd_df["Date"] >= f"{FIXED_NETWORK_START_YEAR}-01-01"].reset_index(drop=True)

print(f"Pre-{FIXED_NETWORK_START_YEAR} HRDs (full network):     {len(pre_1880_hrds):,}")
print(f"{FIXED_NETWORK_START_YEAR}+ HRDs (constant network):    {len(post_1880_hrds):,}")

pre_1880_results = run_blob_ratio_for_period(
    pre_1880_hrds, FULL_NETWORK_FILE, NSW_BOUNDARY_FILE, GRID_FILE
)
post_1880_results = run_blob_ratio_for_period(
    post_1880_hrds, CONSTANT_NETWORK_FILE, NSW_BOUNDARY_FILE, GRID_FILE
)

pre_1880_results["Network"] = "Full (available)"
post_1880_results["Network"] = "Constant (fixed)"

ratio_df = pd.concat([pre_1880_results, post_1880_results], ignore_index=True).sort_values("Date")
ratio_df.head()

## Step 3 — Determine the 16% WRx/IRx threshold

> We determined the 16% threshold by calculating the mean of all ratio values from the 17,667 HRDs identified
> between 1858–2024.

In [ ]:
annual_mean_ratio = ratio_df.dropna(subset=["Blob_Ratio"]).groupby(ratio_df["Date"].dt.year)["Blob_Ratio"].mean()
WRX_THRESHOLD = annual_mean_ratio.mean()

print(f"Total HRDs with a valid ratio: {ratio_df['Blob_Ratio'].notna().sum():,}")
print(f"Derived WRx/IRx threshold (mean of annual mean ratios): {WRX_THRESHOLD:.4f}  (manuscript reports 0.16)")

## Step 4 — Classify each HRD as WRx or IRx

In [ ]:
ratio_df["Type"] = np.where(ratio_df["Blob_Ratio"] >= WRX_THRESHOLD, "WRx", "IRx")

wrx_count = (ratio_df["Type"] == "WRx").sum()
irx_count = (ratio_df["Type"] == "IRx").sum()
print(f"WRx days: {wrx_count:,}  (manuscript reports 7,399)")
print(f"IRx days: {irx_count:,}")

ratio_df.to_excel(OUTPUT_FILE, index=False, sheet_name="HRD_Ratio_Classification")

## Step 5 — Group consecutive WRx (and IRx) days into events

> We grouped consecutive WRx days and treated them as a single widespread heavy rainfall event... we treated
> non-consecutive WRx days as individual single-day events. We applied the same classification approach to
> IRx days.

In [ ]:
def group_into_events(df_type):
    """df_type: rows of ratio_df already filtered to a single Type (WRx or IRx), sorted by Date."""
    df_type = df_type.sort_values("Date").reset_index(drop=True).copy()
    df_type["Date_diff"] = df_type["Date"].diff().dt.days
    df_type["Event_ID"] = (df_type["Date_diff"] > 1).cumsum()
    return df_type

wrx_df = group_into_events(ratio_df[ratio_df["Type"] == "WRx"])
irx_df = group_into_events(ratio_df[ratio_df["Type"] == "IRx"])

event_durations = wrx_df.groupby("Event_ID")["Date"].agg(["min", "max", "count"]).reset_index()
event_durations.rename(columns={"count": "Duration_days"}, inplace=True)

print(f"WRx events: {event_durations.shape[0]:,}  (single-day + multi-day)")
event_durations["Duration_days"].value_counts().sort_index()

## Step 6 — Classify WRx events as Coastal / Inland / Mixed

> We designated a WRx event as coastal if all of its constituent WRx days recorded rainfall exclusively at
> stations east of the GDR... inland if... exclusively west... and as mixed if rainfall occurred on both sides.

In [ ]:
coastal_gdf = gpd.read_file(COASTAL_POLYGON_FILE)
full_network_df = pd.read_parquet(FULL_NETWORK_FILE)
full_network_df["Date"] = pd.to_datetime(full_network_df["Date"], errors="coerce")

stations_gdf = gpd.GeoDataFrame(
    full_network_df,
    geometry=gpd.points_from_xy(full_network_df["Lon"], full_network_df["Lat"]),
    crs=coastal_gdf.crs,
)

def classify_day_side(date):
    """Coastal / Inland / Mix based on which side of the GDR recorded rainfall on this day."""
    daily = stations_gdf[(stations_gdf["Date"] == date) & (stations_gdf["Rainfall"] > 0)]
    if daily.empty:
        return "No Data"
    inside = daily.geometry.apply(lambda pt: coastal_gdf.contains(pt).any())
    if inside.all():
        return "Coastal"
    elif inside.any():
        return "Mix"
    return "Inland"

wrx_df["Day_Type"] = wrx_df["Date"].apply(classify_day_side)

def classify_event_side(day_types):
    types = set(day_types) - {"No Data"}
    if types == {"Coastal"}:
        return "Coastal"
    if types == {"Inland"}:
        return "Inland"
    return "Mixed"

event_side = wrx_df.groupby("Event_ID")["Day_Type"].apply(classify_event_side).reset_index(name="Event_Type")
event_durations = event_durations.merge(event_side, on="Event_ID", how="left")

print(event_durations["Event_Type"].value_counts())
print("\nPer manuscript: analysis focuses on Inland + Mixed WRx events.")

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
    wrx_df.to_excel(writer, sheet_name="WRx_Days", index=False)
    irx_df.to_excel(writer, sheet_name="IRx_Days", index=False)
    event_durations.to_excel(writer, sheet_name="WRx_Events", index=False)

## Step 7 — Weekday bias QC check (Monday-high / weekend-low)

> we conducted an additional quality control check to investigate whether a bias existed toward unusually high
> counts of WRx days on Mondays and unusually low counts on Saturdays or Sundays... We found Monday bias more in
> IRx days rather than WRx days occurrences at 30 stations.

In [ ]:
def weekday_bias_check(extreme_days_df, min_years_recorded=30):
    """Per-station chi-square test for uniform weekday distribution, with a directional follow-up
    (Monday-high / Saturday-low / Sunday-low) if the omnibus test is significant."""
    df = extreme_days_df.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
    df["Weekday"] = df["Date"].dt.dayofweek  # 0 = Monday

    results = []
    for station, group in df.groupby("Station Number"):
        years_recorded = group["Date"].dt.year.nunique()
        if years_recorded < min_years_recorded:
            continue

        counts = group["Weekday"].value_counts().reindex(range(7), fill_value=0).values
        total = counts.sum()
        expected = np.ones(7) * (total / 7)
        chi2, p_chi, _, _ = chi2_contingency([counts, expected])

        if p_chi < 0.05:
            p_monday = binomtest(counts[0], total, 1 / 7, alternative="greater").pvalue
            p_sat = binomtest(counts[5], total, 1 / 7, alternative="less").pvalue
            p_sun = binomtest(counts[6], total, 1 / 7, alternative="less").pvalue
            bias_flag = (p_monday < 0.05) or (p_sat < 0.05) or (p_sun < 0.05)
        else:
            p_monday = p_sat = p_sun = np.nan
            bias_flag = False

        results.append({
            "Station Number": station, "Years Recorded": years_recorded, "Chi2 p-value": p_chi,
            "Monday Count": counts[0], "Saturday Count": counts[5], "Sunday Count": counts[6],
            "Monday-Weekend Bias": bias_flag,
        })
    return pd.DataFrame(results).sort_values("Chi2 p-value")

# NOTE: run separately on WRx-day station records and IRx-day station records to compare bias rates,
# per the manuscript's finding that bias was more prevalent in IRx days.
# wrx_station_days = <per-station rainfall records restricted to WRx dates>
# bias_results = weekday_bias_check(wrx_station_days)
# print(f"Stations with Monday-weekend bias: {bias_results['Monday-Weekend Bias'].sum()}")